# 作业二 · Benchmark

在 Colab 选择 GPU 运行时，或在 VS Code 选择 Colab GPU 内核。
新会话先运行第 2 节。整模型的 torch.compile 对比在第 7 节，attention 对比在第 6 节。

## 1. 代码包
第 2 节从 Google Drive 的 `MyDrive/cs336/assignment2_colab.zip` 读取代码。

## 2. 解压代码，准备环境
挂载 Drive 保存结果；实际 Python、PyTorch 和显卡信息保存到 `environment.json`。


In [1]:
from google.colab import drive
from pathlib import Path
import json, os, subprocess, sys, zipfile
from datetime import datetime

drive.mount('/content/drive')
PROJECT = Path('/content/assignment2-systems')
with zipfile.ZipFile('/content/drive/MyDrive/cs336/assignment2_colab.zip') as archive:
    archive.extractall(PROJECT)
os.chdir(PROJECT)
os.environ['PYTHONPATH'] = str(PROJECT / 'cs336-basics')
RESULTS = Path('/content/drive/MyDrive/cs336/assignment2') / datetime.now().strftime('%Y%m%d_%H%M%S')
RESULTS.mkdir(parents=True)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'einops==0.8.1', 'einx==0.3.0', 'jaxtyping==0.3.2', 'pandas'], check=True,
)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--no-deps',
     '--ignore-requires-python', '-e', str(PROJECT)], check=True,
)
import torch
RUN = [sys.executable]
environment = dict(
    python=sys.version.split()[0], torch=torch.__version__, cuda=torch.version.cuda,
    gpu=torch.cuda.get_device_name(0),
    vram_gib=torch.cuda.get_device_properties(0).total_memory / 2**30,
)
(RESULTS / 'environment.json').write_text(json.dumps(environment, indent=2))
print(environment)
print('结果目录:', RESULTS)

Mounted at /content/drive
{'python': '3.13.16', 'torch': '2.14.1+cu130', 'cuda': '13.0', 'gpu': 'NVIDIA L4', 'vram_gib': 22.0343017578125}
结果目录: /content/drive/MyDrive/cs336/assignment2/20261111_063328


## 3. 实验参数
(b) 使用预热 5 次；(c) 比较 0、1、2 次。默认共 20 组，每组测量 10 次。

In [10]:
MODELS = ['small', 'medium', 'large', 'xl', '2.7B']
WARMUPS = [5, 0, 1, 2]
CONTEXT_LENGTH = 128
MEASUREMENT_STEPS = 10

## 4. 批量运行
每组启动独立进程，JSON 和日志直接保存到 Drive；失败时显示日志并停止。

In [11]:
for model in MODELS:
    for warmup in WARMUPS:
        name = f'{model}_w{warmup}'
        command = RUN + ['-m', 'cs336_systems.benchmark',
            '--model-size', model, '--context-length', str(CONTEXT_LENGTH),
            '--batch-size', '4', '--vocab-size', '10000',
            '--device', 'cuda', '--dtype', 'float32', '--mode', 'forward-backward',
            '--warmup-steps', str(warmup), '--measurement-steps', str(MEASUREMENT_STEPS),
            '--output', str(RESULTS / f'{name}.json')]
        result = subprocess.run(command, capture_output=True, text=True)
        (RESULTS / f'{name}.log').write_text(result.stdout + result.stderr)
        if result.returncode:
            print(result.stdout + result.stderr)
            result.check_returncode()
        print(name, '完成', flush=True)

medium_w1 完成
medium_w2 完成
large_w5 完成
large_w0 完成
large_w1 完成
large_w2 完成
xl_w5 完成
xl_w0 完成
xl_w1 完成
xl_w2 完成
2.7B_w5 完成
2.7B_w0 完成
2.7B_w1 完成
2.7B_w2 完成


In [ ]:
print(result.stdout)
print(result.stderr)

## 5. 汇总结果
单位为毫秒。表格保存为 `summary.csv`，逐次测量在各组 JSON 的 `measurements` 中。

In [12]:
import pandas as pd

rows = []
for model in MODELS:
    for warmup in WARMUPS:
        data = json.loads((RESULTS / f'{model}_w{warmup}.json').read_text())
        row = {'model': model, 'context': data['config']['context_length'], 'warmup': warmup}
        for stage, timing in data['summary'].items():
            row[f'{stage}_ms'] = timing['mean_seconds'] * 1000
            row[f'{stage}_std_ms'] = timing['std_seconds'] * 1000
        rows.append(row)
summary = pd.DataFrame(rows)
summary.to_csv(RESULTS / 'summary.csv', index=False)
display(summary.round(3))

,model,context,warmup,forward_ms,forward_std_ms,loss_ms,loss_std_ms,backward_ms,backward_std_ms
0,small,128,5,13.465,0.165,0.090,0.001,14.293,0.091
1,small,128,0,46.174,103.588,1.325,3.894,20.804,19.767
2,small,128,1,13.325,0.418,0.092,0.005,13.637,0.638
3,small,128,2,13.469,0.126,0.090,0.001,14.435,0.043
4,medium,128,5,26.643,0.185,0.093,0.003,32.658,0.021
5,medium,128,0,58.155,99.133,0.927,2.627,38.585,18.725
6,medium,128,1,26.218,0.220,0.094,0.007,32.659,0.085
7,medium,128,2,26.307,0.128,0.094,0.004,32.678,0.030
8,large,128,5,39.734,0.420,0.091,0.002,67.776,0.450
9,large,128,0,62.723,70.972,0.934,2.654,73.368,16.864


## 6. Attention · 普通版与 torch.compile

两格可分别运行。每版固定 batch=8、FP32、因果掩码，对 4 种维度（16、32、64、128）和 5 种序列长度（256、1024、4096、8192、16384）逐组预热 5 次，再测 100 次前向和反向。编译发生在预热阶段。

结果逐组写入 Drive 的本次实验目录。显存不足的组合标记为 `OOM`。

In [2]:
# 普通版
subprocess.run(
    RUN + ['-m', 'cs336_systems.attention_benchmark',
           '--output', str(RESULTS / 'attention_eager.csv')],
    check=True,
)

CompletedProcess(args=['/usr/bin/python3', '-m', 'cs336_systems.attention_benchmark', '--output', '/content/drive/MyDrive/cs336/assignment2/20261111_063328/attention_eager.csv'], returncode=0)

In [3]:
# torch.compile 版
subprocess.run(
    RUN + ['-m', 'cs336_systems.attention_benchmark', '-compile',
           '--output', str(RESULTS / 'attention_compiled.csv')],
    check=True,
)

CompletedProcess(args=['/usr/bin/python3', '-m', 'cs336_systems.attention_benchmark', '-compile', '--output', '/content/drive/MyDrive/cs336/assignment2/20261111_063328/attention_compiled.csv'], returncode=0)

### 对比结果

`forward_ms` 和 `backward_ms` 是平均耗时，`before_backward_mib` 是反向前已分配的显存。加速比为普通版耗时除以编译版耗时，大于 1 表示编译版更快。

In [4]:
import pandas as pd

eager = pd.read_csv(RESULTS / 'attention_eager.csv')
compiled = pd.read_csv(RESULTS / 'attention_compiled.csv')
comparison = eager.merge(compiled, on=['d_model', 'seq_len'], suffixes=('_eager', '_compiled'))
for stage in ['forward', 'backward']:
    comparison[f'{stage}_speedup'] = comparison[f'{stage}_ms_eager'] / comparison[f'{stage}_ms_compiled']

comparison.to_csv(RESULTS / 'attention_comparison.csv', index=False)
display(comparison.round(2))

,d_model,seq_len,forward_ms_eager,backward_ms_eager,before_backward_mib_eager,status_eager,forward_ms_compiled,backward_ms_compiled,before_backward_mib_compiled,status_compiled,forward_speedup,backward_speedup
0,16,256,0.50,0.94,20.96,OK,0.35,0.47,20.97,OK,1.46,2.02
1,16,1024,1.31,4.08,83.84,OK,0.80,1.98,83.88,OK,1.63,2.06
2,16,4096,31.82,73.41,1066.62,OK,10.76,30.10,1066.75,OK,2.96,2.44
3,16,8192,125.57,289.65,4197.00,OK,49.59,118.57,4197.25,OK,2.53,2.44
4,16,16384,NaN,NaN,NaN,OOM,NaN,NaN,NaN,OOM,NaN,NaN
5,32,256,0.50,0.92,21.59,OK,0.38,0.49,21.59,OK,1.30,1.86
6,32,1024,1.36,4.19,86.34,OK,0.90,2.04,86.38,OK,1.51,2.05
7,32,4096,31.97,73.82,1076.62,OK,13.41,31.90,1076.75,OK,2.38,2.31
8,32,8192,125.81,291.95,4217.00,OK,65.74,131.27,4217.25,OK,1.91,2.22
9,32,16384,NaN,NaN,NaN,OOM,NaN,NaN,NaN,OOM,NaN,NaN


## 7. 整个 Transformer · 普通版与 torch.compile

在同一张 GPU 上运行下面两格。默认使用 `small` 模型、context=128、batch=4 和 FP32。两版均预热 5 步，再测 10 步。`forward` 测纯推理前向，`train` 测前向、loss、反向和 AdamW 更新；`--compile` 编译模型，loss 和 AdamW 沿用原实现。

In [5]:
model_size = 'small'
context_length = 128
transformer_dir = RESULTS / f'transformer_{model_size}_ctx{context_length}'
command = RUN + ['-m', 'cs336_systems.benchmark', '--model-size', model_size,
                 '--context-length', str(context_length), '--batch-size', '4', '--dtype', 'float32',
                 '--warmup-steps', '5', '--measurement-steps', '10']

for version, flags in [('eager', []), ('compiled', ['--compile'])]:
    for mode in ['forward', 'train']:
        output = transformer_dir / f'{version}_{mode}.json'
        print(f'{version} · {mode}', flush=True)
        subprocess.run(command + flags + ['--mode', mode, '--output', str(output)], check=True)

eager · forward
eager · train
compiled · forward
compiled · train


In [6]:
import pandas as pd

columns = {}
for version in ['eager', 'compiled']:
    forward = json.loads((transformer_dir / f'{version}_forward.json').read_text())['summary']
    train = json.loads((transformer_dir / f'{version}_train.json').read_text())['summary']
    stages = {f'train_{name}': stats['mean_seconds'] * 1000 for name, stats in train.items()}
    columns[f'{version}_ms'] = {'inference_forward': forward['forward']['mean_seconds'] * 1000,
                                **stages, 'train_total': sum(stages.values())}

comparison = pd.DataFrame(columns).rename_axis('stage')
comparison['speedup'] = comparison['eager_ms'] / comparison['compiled_ms']
comparison.to_csv(transformer_dir / 'comparison.csv')
display(comparison.round(3))

,eager_ms,compiled_ms,speedup
stage,,,
inference_forward,38.939,15.551,2.504
train_forward,46.360,15.828,2.929
train_loss,0.401,0.487,0.823
train_backward,50.401,37.098,1.359
train_optimizer_step,19.439,20.583,0.944
train_total,116.602,73.997,1.576
